In [ ]:
import matplotlib.pyplot as plt
from matplotlib import patches
from shapely.plotting import plot_polygon
from shapely.wkt import loads as load_wkt
from tspn_bnb2 import Instance, Point, branch_and_bound
from tspn_bnb2.core import parse_site_wkt
from tspn_bnb2.core._tspn_bindings import Trajectory
from tspn_bnb2.misc.plotting import set_bbox

cmap = plt.get_cmap("tab20")

In [ ]:
def plot_traj(ax, instance, trajectory=None, colors=None):
    """Plot trajectory matching root_node_strategies style."""
    sites = instance.sites()
    set_bbox(ax, sites)
    for i, site in enumerate(sites):
        color = colors[i] if colors else cmap(i % cmap.N)
        fc = (*color[:3], 0.5)
        if trajectory and trajectory.distance_site(site) >= 0.0001:
            fc = (0.7, 0.7, 0.7, 0.5)
        poly = load_wkt(str(site))
        plot_polygon(poly, ax=ax, facecolor=fc, edgecolor="black", add_points=False, linewidth=0.5)
    if trajectory:
        xs = [p.x for p in trajectory]
        ys = [p.y for p in trajectory]
        ax.plot(xs, ys, "-", color="black", lw=1, zorder=10)
        ax.plot(xs, ys, "o", markersize=3, color="black", zorder=10)
    ax.set_aspect("equal", "datalim")


decomp_branch_list = [
    parse_site_wkt(
        "POLYGON((1 -3, 0.3 -1.5, 0 -2.3, -0.3 -1.5, -1 -2.8, -1 -1, -0.3 -1, 0.3 -1, 2 -1))"
    ),
    parse_site_wkt("POLYGON((-1 -4, -1 -5, -2 -5))"),
    parse_site_wkt("POLYGON((1 -4, 1 -5, 2 -5))"),
]
decompositions = [
    parse_site_wkt("POLYGON((-0.3 -1, -0.3 -1.5, -1 -2.8, -1 -1))"),
    parse_site_wkt("POLYGON((0.3 -1, 0.3 -1.5, 0 -2.3, -0.3 -1.5, -0.3 -1, 0.3 -1))"),
    parse_site_wkt("POLYGON((1 -3, 0.3 -1.5, 0.3 -1, 2 -1))"),
]
decomp_instance = Instance(decomp_branch_list, False)

# Fixed color assignment: big_poly=0, tri_left=1, tri_right=2
# Parent has [big_poly, tri_left, tri_right]
parent_colors = [cmap(0), cmap(1), cmap(2)]
# Children have [tri_left, tri_right, decomp_piece] — decomp piece inherits big_poly's color
child_colors = [cmap(1), cmap(2), cmap(0)]

print(parent_colors, child_colors)

fig, ax = plt.subplots(2, 3, figsize=(3, 2))
for row in ax:
    for a in row:
        a.axis("off")
        a.set_aspect("equal")


def plt_cb(ecb):
    if ecb.current_node.depth() == 0:
        traj = ecb.get_relaxed_solution().get_trajectory()
        plot_traj(ax[0][1], decomp_instance, traj, colors=parent_colors)


for col in range(3):
    t_inst = Instance([decomp_branch_list[1], decomp_branch_list[2], decompositions[col]], False)
    t_points = [
        [Point(-1, -4), Point(1, -4), Point(-1, -2.8), Point(-1, -4)],
        [Point(-1, -4), Point(1, -4), Point(0, -2.3), Point(-1, -4)],
        [Point(-1, -4), Point(1, -4), Point(1, -3), Point(-1, -4)],
    ]
    t_traj = Trajectory(t_points[col])
    plot_traj(ax[1][col], t_inst, t_traj, colors=child_colors)

branch_and_bound(decomp_instance, plt_cb)
axa = ax[0][1]


def lincomb(a, b, x):
    return a * x + b * (1 - x)


def add_arrow(fig, axa, axb, xya, xyb):
    arrow = patches.ConnectionPatch(
        xya,
        xyb,
        coordsA=axa.transData,
        coordsB=axb.transData,
        color="black",
        arrowstyle="-|>",
        mutation_scale=7.5,
        linewidth=1.25,
    )
    fig.patches.append(arrow)


xlim = axa.get_xlim()
ylim = axa.get_ylim()
start_axa = (
    lincomb(xlim[0], xlim[1], 0.5),
    lincomb(ylim[1], ylim[0], 0.06),
)
for i, axb in enumerate(ax[1]):
    xlim = axb.get_xlim()
    ylim = axb.get_ylim()
    end_axb = (
        lincomb(xlim[0], xlim[1], i / 8 + 0.375),
        lincomb(ylim[1], ylim[0], 0.94),
    )

    add_arrow(fig, axa, axb, start_axa, end_axb)

fig.tight_layout()
fig.savefig("out/decomp_branch_demo.pdf", bbox_inches="tight", facecolor="white")
plt.show()